# Daily Challenge — RAG with LangChain and Hugging Face

## Solution complète et exécutable dans Google Colab

Ce notebook construit un système de **Retrieval-Augmented Generation (RAG)**
avec :

- `databricks/databricks-dolly-15k` comme corpus ;
- `HuggingFaceDatasetLoader` pour créer des documents LangChain ;
- `RecursiveCharacterTextSplitter` pour le chunking ;
- `sentence-transformers/all-MiniLM-L6-v2` pour les embeddings ;
- FAISS comme vector store ;
- `google/flan-t5-small` comme générateur local ;
- `RetrievalQA` comme chaîne de question-réponse.

Aucune clé API n’est nécessaire.

## Objectifs pédagogiques

À la fin du notebook, vous saurez :

1. charger un dataset Hugging Face ;
2. convertir ses lignes en objets `Document` ;
3. nettoyer et filtrer le corpus ;
4. découper les documents en chunks ;
5. calculer des embeddings ;
6. construire un index FAISS ;
7. créer et inspecter un retriever ;
8. préparer un modèle local Hugging Face ;
9. construire une chaîne `RetrievalQA` ;
10. distinguer QA extractive et génération RAG.

## Workflow

```text
Dolly 15k
   ↓
Documents LangChain
   ↓
Nettoyage + filtrage
   ↓
Chunks
   ↓
Embeddings MiniLM
   ↓
FAISS
   ↑
Question → Retriever → Contexte → FLAN-T5 → Réponse
```

Une réponse RAG dépend de deux étapes différentes :

\[
\text{Qualité finale}
=
\text{qualité du retrieval}
+
\text{fidélité du générateur}
\]

Il faut donc inspecter les passages récupérés avant d’évaluer le modèle.

## Corrections apportées au code de l’énoncé

Plusieurs imports ont changé dans les versions modernes de LangChain :

| Ancien import | Import utilisé |
|---|---|
| `langchain.document_loaders` | `langchain_community.document_loaders` |
| `langchain.text_splitter` | `langchain_text_splitters` |
| `langchain.embeddings` | `langchain_huggingface` |
| `langchain.vectorstores` | `langchain_community.vectorstores` |
| `from langchain import HuggingFacePipeline` | `langchain_huggingface` |
| `langchain.chains.RetrievalQA` | `langchain_classic.chains` |

Autres corrections :

- le nom correct du modèle est `all-MiniLM-L6-v2` ;
- `qa.invoke({"query": question})` remplace l’ancien usage de `run` ;
- un pipeline `question-answering` extractif n’est pas un LLM génératif ;
- `Intel/dynamic_tinybert` sera montré séparément ;
- la chaîne RAG utilisera FLAN-T5, compatible avec `HuggingFacePipeline`.

## 1. Installation

In [ ]:
%pip install -q \
    "torch>=2.2,<3.0" \
    "transformers>=4.45,<5.0" \
    "sentence-transformers>=3.0,<6.0" \
    "datasets>=3.0,<5.0" \
    "faiss-cpu>=1.8,<2.0" \
    "langchain-core>=1.0,<2.0" \
    "langchain-community>=0.4,<0.5" \
    "langchain-classic>=1.0,<2.0" \
    "langchain-huggingface>=1.0,<2.0" \
    "langchain-text-splitters>=1.0,<2.0" \
    "accelerate>=1.0,<2.0" \
    "sentencepiece>=0.2,<1.0" \
    "pandas>=2.0,<3.0"

In [ ]:
import hashlib
import importlib.metadata as metadata
import re
import textwrap
import warnings
from collections import Counter
from time import perf_counter
from typing import Dict, List, Optional

import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from transformers import (
    AutoModel,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline,
)

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQA

warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Appareil :", DEVICE)
print("\nVersions principales :")

for package_name in [
    "torch",
    "transformers",
    "sentence-transformers",
    "datasets",
    "faiss-cpu",
    "langchain-community",
    "langchain-classic",
    "langchain-huggingface",
]:
    try:
        print(f"- {package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"- {package_name}: introuvable")

## 2. Configuration

In [ ]:
DATASET_NAME = "databricks/databricks-dolly-15k"
PAGE_CONTENT_COLUMN = "context"

EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL_ID = "google/flan-t5-small"
EXTRACTIVE_QA_MODEL_ID = "Intel/dynamic_tinybert"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
DEFAULT_K = 4

# True : indexation plus rapide sur CPU.
# False : indexation de tous les contextes valides.
FAST_MODE = True
FAST_MODE_MAX_DOCUMENTS = 2500

print("Configuration prête.")

## 3. Introduction à Hugging Face Transformers

Un tokenizer transforme le texte en identifiants numériques. BERT produit
ensuite un vecteur pour chaque token.

La forme de `last_hidden_state` est :

```text
batch_size × nombre_de_tokens × dimension_cachée
```

Ces embeddings sont contextuels : le vecteur d’un mot dépend des autres mots de
la phrase.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_model.to(DEVICE)
bert_model.eval()

sample_text = "Hello Hugging Face!"

sample_inputs = bert_tokenizer(
    sample_text,
    return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    sample_outputs = bert_model(**sample_inputs)

print(
    "Forme de last_hidden_state :",
    tuple(sample_outputs.last_hidden_state.shape),
)
print(
    "Dimension d’un embedding de token :",
    sample_outputs.last_hidden_state.shape[-1],
)

### Embeddings de tokens et embeddings de documents

`last_hidden_state` contient un vecteur par token. Pour la recherche, nous avons
besoin d’un seul vecteur par chunk.

Sentence Transformers applique un pooling et un entraînement spécialisé pour
la similarité sémantique. Nous utiliserons donc MiniLM pour le retriever.

## 4. Chargement du dataset

In [ ]:
loading_start = perf_counter()

try:
    loader = HuggingFaceDatasetLoader(
        DATASET_NAME,
        PAGE_CONTENT_COLUMN,
    )
    raw_documents = loader.load()
    loading_method = "LangChain HuggingFaceDatasetLoader"

except Exception as loader_error:
    print("Échec du loader LangChain :", repr(loader_error))
    print("Utilisation du fallback datasets.load_dataset.")

    hf_dataset = load_dataset(DATASET_NAME, split="train")
    raw_documents = []

    for row_index, row in enumerate(hf_dataset):
        row_metadata = {
            key: value
            for key, value in row.items()
            if key != PAGE_CONTENT_COLUMN
        }
        row_metadata["row_index"] = row_index

        raw_documents.append(
            Document(
                page_content=str(row.get(PAGE_CONTENT_COLUMN, "") or ""),
                metadata=row_metadata,
            )
        )

    loading_method = "Hugging Face Datasets fallback"

loading_elapsed = perf_counter() - loading_start

print("Méthode :", loading_method)
print("Nombre de lignes :", len(raw_documents))
print(f"Temps : {loading_elapsed:.2f} secondes")
print("\nPremier document brut :")
print(raw_documents[0])

### Pourquoi certains contextes sont-ils vides ?

Dolly 15k est un dataset d’instruction-following. Certaines catégories, comme
le brainstorming ou l’open QA, ne nécessitent pas de passage de référence.

Les catégories closed QA, summarization et information extraction contiennent
plus souvent un `context`. Nous devons donc filtrer les chaînes vides avant
l’indexation.

In [ ]:
category_counts = Counter(
    document.metadata.get("category", "unknown")
    for document in raw_documents
)

display(
    pd.DataFrame(
        category_counts.most_common(),
        columns=["category", "number_of_rows"],
    )
)

## 5. Nettoyage, filtrage et déduplication

In [ ]:
def clean_context(text: str) -> str:
    """Nettoie légèrement le texte sans modifier son sens."""
    text = str(text or "")
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


valid_documents: List[Document] = []
seen_hashes = set()

for row_index, document in enumerate(raw_documents):
    cleaned_text = clean_context(document.page_content)

    if not cleaned_text:
        continue

    context_hash = hashlib.sha1(
        cleaned_text.encode("utf-8")
    ).hexdigest()

    if context_hash in seen_hashes:
        continue

    seen_hashes.add(context_hash)

    metadata_copy = dict(document.metadata)
    metadata_copy.setdefault("row_index", row_index)
    metadata_copy["context_hash"] = context_hash[:16]
    metadata_copy["dataset"] = DATASET_NAME

    valid_documents.append(
        Document(
            page_content=cleaned_text,
            metadata=metadata_copy,
        )
    )

print("Lignes brutes :", len(raw_documents))
print("Contextes uniques et non vides :", len(valid_documents))

### Mode rapide

Le mode rapide limite le nombre de documents indexés, mais place en priorité les
passages contenant `cheese` ou `cheesemaking`. Cela permet de tester la question
demandée sans exclure accidentellement un passage pertinent.

Passez `FAST_MODE` à `False` pour indexer tout le corpus valide.

In [ ]:
priority_terms = ("cheesemaking", "cheese making", "cheese")

priority_documents = [
    document
    for document in valid_documents
    if any(
        term in document.page_content.lower()
        for term in priority_terms
    )
]

priority_hashes = {
    document.metadata["context_hash"]
    for document in priority_documents
}

remaining_documents = [
    document
    for document in valid_documents
    if document.metadata["context_hash"] not in priority_hashes
]

if FAST_MODE:
    documents = (
        priority_documents + remaining_documents
    )[:FAST_MODE_MAX_DOCUMENTS]
else:
    documents = valid_documents

print("Mode rapide :", FAST_MODE)
print("Documents prioritaires :", len(priority_documents))
print("Documents retenus :", len(documents))

if priority_documents:
    print("\nExtrait lié au fromage :")
    print(priority_documents[0].page_content[:700])

## 6. Chunking avec `RecursiveCharacterTextSplitter`

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""],
)

docs = text_splitter.split_documents(documents)

for chunk_id, document_chunk in enumerate(docs):
    document_chunk.metadata["chunk_id"] = chunk_id
    document_chunk.metadata["chunk_size_setting"] = CHUNK_SIZE
    document_chunk.metadata["chunk_overlap_setting"] = CHUNK_OVERLAP

print("Documents avant chunking :", len(documents))
print("Chunks après découpage :", len(docs))
print("\nPremier chunk :")
print(docs[0])

In [ ]:
chunk_lengths = [len(document.page_content) for document in docs]

display(
    pd.Series(chunk_lengths).describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
    ).to_frame("characters")
)

assert len(docs) > 0
assert all(document.page_content.strip() for document in docs)
assert max(chunk_lengths) <= CHUNK_SIZE

## 7. Embeddings

`all-MiniLM-L6-v2` produit un vecteur dense de 384 dimensions pour chaque
chunk. Les embeddings sont normalisés pour faciliter la comparaison
sémantique.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={"device": DEVICE},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,
    },
)

test_embedding = embeddings.embed_query(
    "This is a test document."
)

print("Dimension :", len(test_embedding))
print("Trois premières valeurs :", test_embedding[:3])

## 8. Création du vector store FAISS

In [ ]:
indexing_start = perf_counter()

db = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
)

indexing_elapsed = perf_counter() - indexing_start

print(f"Index construit en {indexing_elapsed:.2f} secondes")
print("Chunks indexés :", len(docs))

In [ ]:
FAISS_INDEX_PATH = "dolly_faiss_index"
db.save_local(FAISS_INDEX_PATH)

print("Index sauvegardé dans :", FAISS_INDEX_PATH)

## 9. Retriever et sanity check

Le retriever sélectionne les chunks, mais ne produit aucune réponse.

Il faut lire les résultats avant de connecter le modèle génératif. Si le bon
passage n’est pas présent, le problème vient du retrieval ou du corpus.

In [ ]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": DEFAULT_K},
)


def inspect_retrieval(
    question: str,
    k: Optional[int] = None,
) -> List[Document]:
    """Affiche les chunks récupérés pour une question."""
    if not question.strip():
        raise ValueError("La question ne peut pas être vide.")

    active_retriever = retriever

    if k is not None:
        active_retriever = db.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k},
        )

    retrieved_documents = active_retriever.invoke(question)

    rows = []

    for rank, document in enumerate(retrieved_documents, start=1):
        rows.append(
            {
                "rank": rank,
                "category": document.metadata.get("category"),
                "chunk_id": document.metadata.get("chunk_id"),
                "start_index": document.metadata.get("start_index"),
                "instruction": str(
                    document.metadata.get("instruction", "")
                )[:180],
                "preview": textwrap.shorten(
                    document.page_content,
                    width=500,
                    placeholder=" ...",
                ),
            }
        )

    display(pd.DataFrame(rows))
    return retrieved_documents

In [ ]:
exercise_question = "What is cheesemaking?"

cheese_retrieved_documents = inspect_retrieval(
    exercise_question,
    k=DEFAULT_K,
)

contains_cheese_evidence = any(
    any(
        term in document.page_content.lower()
        for term in priority_terms
    )
    for document in cheese_retrieved_documents
)

print(
    "Un passage explicitement lié au fromage est récupéré :",
    contains_cheese_evidence,
)

### Question de contrôle issue du dataset

Pour disposer d’un test dont le contexte est garanti dans le corpus, nous
sélectionnons une instruction Dolly associée à l’un des documents indexés.

La réponse Dolly est conservée uniquement comme référence d’évaluation.

In [ ]:
grounded_document = next(
    (
        document
        for document in documents
        if str(document.metadata.get("instruction", "")).strip()
    ),
    None,
)

if grounded_document is None:
    raise RuntimeError(
        "Aucune instruction associée à un contexte n’a été trouvée."
    )

grounded_question = str(
    grounded_document.metadata["instruction"]
).strip()

grounded_reference = str(
    grounded_document.metadata.get("response", "")
).strip()

print("Question :")
print(grounded_question)

print("\nRéponse de référence Dolly :")
print(grounded_reference)

print("\nPassages récupérés :")
grounded_retrieved_documents = inspect_retrieval(
    grounded_question,
    k=DEFAULT_K,
)

### Effet de `k`

In [ ]:
for k_value in [2, 4, 6]:
    print("\n" + "=" * 100)
    print("k =", k_value)
    _ = inspect_retrieval(
        grounded_question,
        k=k_value,
    )

## 10. Préparation du générateur

### Pourquoi TinyBERT n’est pas utilisé comme LLM dans `RetrievalQA`

Un pipeline `question-answering` extractif reçoit directement :

```python
question=...
context=...
```

Il retourne un span extrait du contexte.

`RetrievalQA` attend plutôt un modèle qui reçoit un prompt complet et génère du
texte. Nous utilisons donc `google/flan-t5-small` avec la tâche
`text2text-generation`.

TinyBERT sera testé séparément dans une section ultérieure.

In [ ]:
generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_ID
)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATOR_MODEL_ID
)

pipeline_device = 0 if torch.cuda.is_available() else -1

generation_pipeline = pipeline(
    task="text2text-generation",
    model=generator_model,
    tokenizer=generator_tokenizer,
    device=pipeline_device,
    max_new_tokens=160,
    do_sample=False,
    truncation=True,
)

llm = HuggingFacePipeline(
    pipeline=generation_pipeline,
)

print(
    "Générateur chargé sur",
    "GPU" if pipeline_device == 0 else "CPU",
)

## 11. Construction de `RetrievalQA`

Nous utilisons `chain_type="stuff"` : les chunks sont réunis dans un prompt
unique. Cette méthode est simple et adaptée à un petit modèle local.

Le mode `refine` demandé dans l’énoncé est plus coûteux, car il effectue
plusieurs appels successifs au modèle.

In [ ]:
RAG_PROMPT_TEMPLATE = """
Use only the context below to answer the question.

If the answer is not contained in the context, respond exactly:
I do not know based on the retrieved documents.

Give a concise and factual answer.

Context:
{context}

Question:
{question}

Answer:
""".strip()

rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt},
)

print("Chaîne RetrievalQA prête.")

## 12. Fonction RAG réutilisable

In [ ]:
def ask_rag(
    question: str,
    show_sources: bool = True,
) -> Dict:
    """Exécute le pipeline RAG et affiche les sources."""
    clean_question = question.strip()

    if not clean_question:
        raise ValueError("La question ne peut pas être vide.")

    result = qa.invoke({"query": clean_question})

    print("=" * 100)
    print("QUESTION")
    print(clean_question)

    print("\nRÉPONSE")
    print(result["result"])

    if show_sources:
        print("\nSOURCES")

        for rank, document in enumerate(
            result["source_documents"],
            start=1,
        ):
            print(
                f"\n[{rank}] "
                f"category={document.metadata.get('category')} "
                f"| chunk_id={document.metadata.get('chunk_id')} "
                f"| start={document.metadata.get('start_index')}"
            )

            print(
                textwrap.shorten(
                    document.page_content,
                    width=650,
                    placeholder=" ...",
                )
            )

    return result

### Test demandé

In [ ]:
cheese_result = ask_rag(
    "What is cheesemaking?",
    show_sources=True,
)

Si le corpus ne contient pas une définition pertinente, la bonne réaction du
système est de reconnaître que la réponse n’est pas disponible. Un RAG n’est
pas censé inventer une information absente de sa base documentaire.

### Test fondé sur une ligne Dolly

In [ ]:
grounded_result = ask_rag(
    grounded_question,
    show_sources=True,
)

print("\n" + "=" * 100)
print("RÉPONSE DE RÉFÉRENCE DOLLY")
print(grounded_reference)

## 13. Démonstration séparée de TinyBERT extractif

Le workflow est différent :

```text
Question → FAISS → meilleur chunk → TinyBERT → span extrait
```

TinyBERT peut être utile lorsque la réponse doit apparaître littéralement dans
le passage.

In [ ]:
extractive_tokenizer = AutoTokenizer.from_pretrained(
    EXTRACTIVE_QA_MODEL_ID
)

extractive_model = AutoModelForQuestionAnswering.from_pretrained(
    EXTRACTIVE_QA_MODEL_ID
)

extractive_pipeline = pipeline(
    task="question-answering",
    model=extractive_model,
    tokenizer=extractive_tokenizer,
    device=pipeline_device,
)

top_document = grounded_retrieved_documents[0]

extractive_result = extractive_pipeline(
    question=grounded_question,
    context=top_document.page_content,
)

print("Question :", grounded_question)
print("\nRésultat extractif :")
print(extractive_result)

print("\nContexte utilisé :")
print(top_document.page_content[:1000])

### Comparaison

| Aspect | RAG avec FLAN-T5 | TinyBERT extractif |
|---|---|---|
| Sortie | texte généré | span du contexte |
| Reformulation | oui | non |
| Synthèse multi-chunks | possible | limitée |
| Risque d’hallucination | présent | plus faible |
| Compatible comme LLM `RetrievalQA` | oui | non |

## 14. Méthode académique de diagnostic

### Étape 1 — Vérifier les données

- le champ `context` est-il rempli ?
- le corpus contient-il la réponse ?
- les textes sont-ils dupliqués ou obsolètes ?

### Étape 2 — Vérifier le chunking

- la réponse est-elle coupée entre deux chunks ?
- les chunks sont-ils trop grands ?
- l’overlap crée-t-il trop de redondance ?

### Étape 3 — Vérifier le retrieval

- le bon passage apparaît-il dans le top-k ?
- `k` est-il trop faible ?
- le modèle d’embedding correspond-il au domaine ?

### Étape 4 — Vérifier la génération

- le bon passage est présent mais ignoré ?
- le prompt autorise-t-il implicitement l’invention ?
- le contexte est-il tronqué ?

### Étape 5 — Évaluer

Pour une évaluation complète, utiliser notamment :

- Recall@k ;
- MRR ;
- exactitude de la réponse ;
- fidélité au contexte ;
- exactitude des citations ;
- latence et utilisation mémoire.

## 15. Limites

1. Dolly 15k est un dataset d’instructions, pas une encyclopédie exhaustive.
2. Beaucoup de lignes ne possèdent aucun contexte.
3. FLAN-T5-small est un modèle pédagogique limité.
4. FAISS est ici un index local, sans gestion des permissions.
5. Le prompt réduit les hallucinations sans les supprimer.
6. `RetrievalQA` est une API historique conservée pour respecter l’exercice.
7. Le mode rapide n’indexe pas tous les contextes valides.

## 16. Mode complet

Pour indexer tous les contextes valides :

```python
FAST_MODE = False
```

Puis réexécutez les cellules à partir de la construction du corpus.

En production, l’indexation est généralement séparée du traitement des
questions. L’index est calculé une fois, sauvegardé, puis rechargé.

## Conclusion

Le pipeline réalisé est :

```text
Dataset → Documents → Chunks → Embeddings → FAISS
                                           ↓
Question → Retriever → Contexte → FLAN-T5 → Réponse
```

Les points essentiels sont :

- les documents doivent être nettoyés avant l’indexation ;
- le chunking influence directement la recherche ;
- MiniLM permet une recherche sémantique ;
- FAISS sélectionne les passages pertinents ;
- le retrieval doit être inspecté avant la génération ;
- un modèle extractif n’est pas un générateur ;
- les sources sont indispensables pour auditer les réponses.

> Un RAG fiable ne dépend pas uniquement du LLM. Il dépend de toute la chaîne :
> données, chunking, embeddings, retrieval, prompt et génération.

## Références

- Dataset :
  https://huggingface.co/datasets/databricks/databricks-dolly-15k
- Hugging Face Dataset Loader :
  https://docs.langchain.com/oss/python/integrations/document_loaders/hugging_face_dataset
- FAISS avec LangChain :
  https://docs.langchain.com/oss/python/integrations/vectorstores/faiss
- MiniLM :
  https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- FLAN-T5-small :
  https://huggingface.co/google/flan-t5-small
- Intel Dynamic TinyBERT :
  https://huggingface.co/Intel/dynamic_tinybert